# 5.3 · K 近邻分类 / K-Nearest Neighbors (KNN)

> **课程定位 / Where this fits**
> 5.1/5.2 是**参数模型**: 训练时学一组权重, 之后丢掉数据。KNN 是**非参数惰性(lazy)**模型的代表: **不训练**, 把全部训练数据记住, 预测时找最近的 K 个邻居投票。4.10 讲过 KNN 回归, 这里是分类版。
> KNN is the canonical lazy, non-parametric model: no training, just memorise and vote at predict time.

> 💡 **面试相关 / Interview-relevant**
> - "KNN 训练/预测复杂度" ★★★★（训练 O(1), 预测 O(nd)）
> - "为什么 KNN 必须做特征缩放" ★★★★（距离对量纲敏感）
> - "K 怎么选 / K 大小对偏差方差的影响" ★★★★（小K=高方差）
> - "维度灾难为什么打击 KNN" ★★★★
> - "KD-Tree / Ball-Tree 加速原理" ★★★

---

## 学习目标 / Learning Objectives
1. 理解惰性学习: 训练 O(1), 代价全在预测。
2. 从零实现 KNN 分类(欧氏距离 + 多数投票)。
3. **缩放为何关键** + K 的偏差方差权衡。
4. 维度灾难对 KNN 的打击。
5. 加权投票 + KD-Tree 加速。

## 目录 / TOC
1. [惰性学习与算法 ⭐](#1)
2. [🌸 数据 + 从零实现](#2)
3. [缩放为何关键 ⭐](#3)
4. [选 K: 偏差方差 ⭐](#4)
5. [维度灾难 ⭐](#5)
6. [加权 KNN + 加速](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 惰性学习与算法 ⭐ / Lazy Learning

**算法极简**: 预测 $\mathbf{x}$ 时——
1. 算 $\mathbf{x}$ 到**所有**训练点的距离(常用欧氏 $\|\mathbf{x}-\mathbf{x}_i\|_2$)。
2. 取最近的 **K** 个。
3. **分类**: 多数投票; **回归**(4.10): 取均值。

**复杂度**(面试高频):
- **训练**: $O(1)$ —— 只是存下数据, 不学任何参数 → "惰性"。
- **预测**: $O(nd)$ 每个查询 —— 要扫全部 n 个点、d 维。数据大就慢。

KNN 没有"模型方程", **决策边界由数据本身定义**, 高度非线性、能拟合任意形状——代价是存全部数据 + 预测慢 + 怕高维。


<a id="2"></a>
## 2. 数据 + 从零实现 / Data + From Scratch

继续用 **Iris**(5.2 介绍过): 150 朵花, 3 类, 4 特征。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")

iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
sc = StandardScaler().fit(X_tr)
Xtr, Xte = sc.transform(X_tr), sc.transform(X_te)

def knn_predict(X_train, y_train, X_query, k=5):
    preds = []
    for x in X_query:
        d = np.sqrt(((X_train - x)**2).sum(axis=1))   # 欧氏距离到所有训练点
        idx = np.argsort(d)[:k]                         # 最近 k 个
        preds.append(Counter(y_train[idx]).most_common(1)[0][0])  # 多数投票
    return np.array(preds)

pred = knn_predict(Xtr, y_tr, Xte, k=5)
print(f"从零 KNN (k=5) test 准确率: {(pred == y_te).mean():.3f}")

from sklearn.neighbors import KNeighborsClassifier
sk = KNeighborsClassifier(n_neighbors=5).fit(Xtr, y_tr)
print(f"sklearn KNN  (k=5) test 准确率: {sk.score(Xte, y_te):.3f}  | 预测一致率 {(pred==sk.predict(Xte)).mean():.3f}")


<a id="3"></a>
## 3. 缩放为何关键 ⭐ / Why Scaling Is Critical

KNN 靠**距离**。若一个特征量纲大(如"收入"以元计, 范围 0–100000), 另一个小(如"年龄"0–100), 则距离几乎**完全由大量纲特征主导**, 小特征被淹没。**必须先标准化**(3.4)。这是 KNN 最常见的坑。


In [ ]:
# 故意把一个特征放大 1000 倍, 看不缩放的灾难 / blow up one feature
X_bad = X_tr.copy(); X_bad[:, 0] *= 1000
X_bad_te = X_te.copy(); X_bad_te[:, 0] *= 1000

acc_unscaled = (KNeighborsClassifier(5).fit(X_bad, y_tr).score(X_bad_te, y_te))
acc_scaled = (KNeighborsClassifier(5).fit(
    StandardScaler().fit_transform(X_bad),
    y_tr).score(StandardScaler().fit(X_bad).transform(X_bad_te), y_te))
print(f"某特征×1000 不缩放: 准确率 {acc_unscaled:.3f}  (被该特征绑架)")
print(f"标准化后:          准确率 {acc_scaled:.3f}  (恢复正常)")
print("→ KNN 用前必须缩放; 距离对量纲极敏感")


<a id="4"></a>
## 4. 选 K: 偏差方差权衡 ⭐ / Choosing K

- **K 小(=1)**: 决策边界**极度弯曲**, 跟着每个点走 → **低偏差高方差**, 过拟合, 对噪声敏感。
- **K 大**: 边界**平滑**, 趋向多数类 → **高偏差低方差**, K=n 时退化成"永远预测多数类"。

K 是控制复杂度的旋钮(承接 3.10 偏差方差)。常用**交叉验证**选 K, 且 K 取**奇数**避免二分类平票。


In [ ]:
from sklearn.model_selection import cross_val_score
ks = range(1, 31)
cv_acc = [cross_val_score(KNeighborsClassifier(k), Xtr, y_tr, cv=5).mean() for k in ks]
best_k = list(ks)[int(np.argmax(cv_acc))]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(ks), cv_acc, "o-")
ax.axvline(best_k, color="r", ls="--", label=f"最佳 K={best_k}")
ax.set_xlabel("K"); ax.set_ylabel("5-fold CV 准确率")
ax.set_title("选 K: 太小过拟合(高方差), 太大欠拟合(高偏差)"); ax.legend()
plt.tight_layout(); plt.show()
print(f"CV 最佳 K={best_k}, test 准确率 {KNeighborsClassifier(best_k).fit(Xtr,y_tr).score(Xte,y_te):.3f}")


In [ ]:
# 可视化 K=1 vs K=15 决策边界 / boundary K=1 vs K=15
X2 = Xtr[:, 2:4]
xx, yy = np.meshgrid(np.linspace(X2[:,0].min()-.5, X2[:,0].max()+.5, 300),
                     np.linspace(X2[:,1].min()-.5, X2[:,1].max()+.5, 300))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, k in zip(axes, [1, 15]):
    m = KNeighborsClassifier(k).fit(X2, y_tr)
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="viridis")
    ax.scatter(X2[:,0], X2[:,1], c=y_tr, cmap="viridis", edgecolor="k", s=25)
    ax.set_title(f"K={k}: {'锯齿边界(高方差)' if k==1 else '平滑边界(高偏差)'}")
plt.tight_layout(); plt.show()


<a id="5"></a>
## 5. 维度灾难 ⭐ / Curse of Dimensionality

高维空间里, **所有点彼此都差不多远**——"最近邻"失去意义。随维度增加, 最近点和最远点的距离比趋于 1。这是 KNN(及所有距离方法)的根本弱点。


In [ ]:
# 高维下"最近/最远距离比"趋于1 / near vs far distance ratio
rng = np.random.default_rng(0)
dims = [2, 5, 10, 50, 100, 500]
ratios = []
for d in dims:
    P = rng.random((500, d))
    dist = np.sqrt(((P[0] - P[1:])**2).sum(1))
    ratios.append(dist.min() / dist.max())
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dims, ratios, "o-")
ax.set_xlabel("维度 d"); ax.set_ylabel("最近距离 / 最远距离")
ax.set_title("维度灾难: 高维下所有点几乎等距 → '最近邻'失效")
plt.tight_layout(); plt.show()
print("距离比", [f"{r:.2f}" for r in ratios], "→ 维度越高越接近1, KNN 越失效")
print("应对: 先降维(PCA, Part 7) 或选择性少量特征")


<a id="6"></a>
## 6. 加权 KNN + 加速 / Weighted KNN & Speedups

- **加权投票**: 近邻权重更大(`weights='distance'`, 权 $\propto 1/d$), 缓解 K 偏大时远邻干扰。
- **加速**: 暴力预测 $O(nd)$。**KD-Tree**(低维快)/ **Ball-Tree**(高维稍好)把查询降到约 $O(d\log n)$。sklearn 的 `algorithm='auto'` 自动选。


In [ ]:
import time
print("uniform vs distance 加权:")
for w in ["uniform", "distance"]:
    s = KNeighborsClassifier(15, weights=w).fit(Xtr, y_tr).score(Xte, y_te)
    print(f"  weights={w:<9} 准确率 {s:.3f}")

for algo in ["brute", "kd_tree", "ball_tree"]:
    m = KNeighborsClassifier(5, algorithm=algo).fit(Xtr, y_tr)
    t = time.perf_counter(); m.predict(Xte); dt = (time.perf_counter()-t)*1000
    print(f"  algorithm={algo:<10} 预测 {dt:.2f} ms (结果相同, 仅速度差异)")


<a id="7"></a>
## 7. 小结 / Summary

```
KNN: 惰性非参数; 训练 O(1), 预测 O(nd); 多数投票(分类)/均值(回归 4.10)
必须缩放: 距离对量纲敏感, 大量纲特征会绑架结果
选 K: 小K=高方差(过拟合), 大K=高偏差(欠拟合); CV 选, 取奇数
维度灾难: 高维下点近乎等距, "最近邻"失效 → 先降维
加权投票(1/d) + KD/Ball-Tree 加速
```

### 💡 面试速查
1. **训练 O(1) 预测 O(nd)** —— 惰性, 代价在预测
2. **必须缩放** —— 距离对量纲敏感
3. **K 小过拟合 / K 大欠拟合** —— CV 选 K
4. **维度灾难** —— 高维下 KNN 失效, 先 PCA 降维
5. 加速用 KD-Tree(低维) / Ball-Tree(较高维)

### 下一节
**5.4 朴素贝叶斯**——又一个非参数视角: 用贝叶斯定理(2.8)直接建模 P(类|特征), "朴素"地假设特征条件独立, 文本分类的经典快枪手。
